# Скачивание изображений

В рамках практического задания была проведена работа по скачиванию, сортировке и организации изображений люстр по трём стилям: классический, современный и ретро.

Разделение по стилям в коде :
* "classic chandelier"
* "modern chandelier"
* "retro chandelier"

## Подготовка

Для выполнения задачи были установлены и использованы следующие инструменты:
* requests – для выполнения HTTP-запросов и получения изображений,
* beautifulsoup4 – для парсинга веб-страниц (при необходимости),
* tqdm – для отображения прогресса скачивания.

Создание отдельных папок для каждой категории: загруженные изображения должны сортироваться по папкам, например, chandeliers/classic/, chandeliers/modern/, chandeliers/retro/.

In [ ]:
!pip install requests beautifulsoup4 tqdm

## Поиск и скачивание изображений

Были использованы три источника:
* Pixabay – база бесплатных изображений,
* Pexels – художественные фотографии,
* Unsplash – изображения высокого качества.

Для каждого стиля выполнялись поисковые запросы, после чего изображения скачивались и сохранялись в соответствующие папки. Файлы именовались в формате [стиль]_[номер].jpg.

Загрузка фотографий происходит через API, что является более безопасным и эффективным способом по сравнению с веб-скрейпингом.

Этот код позволяет быстро и автоматически собрать датасет изображений люстр, организованных по стилям, для последующего использования в анализе или машинном обучении.

In [ ]:
import requests
import os
from tqdm import tqdm

# 🔑 API-ключи
API_KEYS = {
    "pixabay": "(Ваш_API_ключ)",
    "pexels": "(Ваш_API_ключ)",
}

# 🏷️ Категории люстр
CATEGORIES = {
    "classic": ["classic chandelier", "crystal chandelier", "antique chandelier"],
    "modern": ["modern chandelier", "contemporary chandelier", "minimalist chandelier"],
    "retro": ["retro chandelier", "vintage chandelier", "industrial chandelier"]
}

NUM_IMAGES = 800  # Количество изображений на категорию

# 📂 Функция создания папок и скачивания изображений
def download_image(url, folder, filename):
    os.makedirs(folder, exist_ok=True)
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        path = os.path.join(folder, filename)
        with open(path, "wb") as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"✅ {filename} загружен!")
    else:
        print(f"❌ Ошибка загрузки {filename}")


# 🔍 Функция поиска изображений с Pixabay
def fetch_pixabay_images(style, queries, folder_path, num_images):
    image_count = 0
    page = 1
    while image_count < num_images:
        for query in queries:
            query_formatted = query.replace(" ", "+")
            url = f"https://pixabay.com/api/?key={API_KEYS['pixabay']}&q={query_formatted}&image_type=photo&per_page=50&page={page}"
            response = requests.get(url)
            try:
                data = response.json()
                if "hits" in data and data["hits"]:
                    for hit in tqdm(data["hits"], desc=f"🔽 Pixabay - {query} (страница {page})"):
                        if image_count >= num_images:
                            break
                        image_url = hit["largeImageURL"]
                        filename = f"{style}_pixabay_{image_count+1}.jpg"
                        download_image(image_url, folder_path, filename)
                        image_count += 1
                else:
                    break
            except Exception as e:
                print(f"❌ Ошибка Pixabay: {e}")
                break
        page += 1


# 🔍 Функция поиска изображений с Pexels
def fetch_pexels_images(style, queries, folder_path, num_images):
    headers = {"Authorization": API_KEYS["pexels"]}
    image_count = 0
    page = 1
    while image_count < num_images:
        for query in queries:
            url = f"https://api.pexels.com/v1/search?query={query}&per_page=50&page={page}"
            response = requests.get(url, headers=headers)
            try:
                data = response.json()
                if "photos" in data and data["photos"]:
                    for photo in tqdm(data["photos"], desc=f"🔽 Pexels - {query} (страница {page})"):
                        if image_count >= num_images:
                            break
                        image_url = photo["src"]["large"]
                        filename = f"{style}_pexels_{image_count+1}.jpg"
                        download_image(image_url, folder_path, filename)
                        image_count += 1
                else:
                    break
            except Exception as e:
                print(f"❌ Ошибка Pexels: {e}")
                break
        page += 1


# 🔄 Запуск загрузки изображений
for style, queries in CATEGORIES.items():
    folder_path = f"chandeliers/{style}"
    fetch_pixabay_images(style, queries, folder_path, NUM_IMAGES // 3)
    fetch_pexels_images(style, queries, folder_path, NUM_IMAGES // 3)

print("🎉 Все изображения загружены и отсортированы по папкам!")

🔽 Pixabay - classic chandelier (страница 1):   2%|▏         | 1/50 [00:00<00:11,  4.24it/s]

✅ classic_pixabay_1.jpg загружен!


🔽 Pixabay - classic chandelier (страница 1):   4%|▍         | 2/50 [00:00<00:13,  3.50it/s]

✅ classic_pixabay_2.jpg загружен!


🔽 Pixabay - classic chandelier (страница 1):   6%|▌         | 3/50 [00:00<00:14,  3.25it/s]

...

✅ retro_pexels_262.jpg загружен!
✅ retro_pexels_263.jpg загружен!


🔽 Pexels - industrial chandelier (страница 2):  28%|██▊       | 14/50 [00:09<00:31,  1.16it/s]

✅ retro_pexels_264.jpg загружен!


🔽 Pexels - industrial chandelier (страница 2):  32%|███▏      | 16/50 [00:10<00:22,  1.48it/s]

✅ retro_pexels_265.jpg загружен!
✅ retro_pexels_266.jpg загружен!
🎉 Все изображения загружены и отсортированы по папкам!


1. Настройки API
- В словаре API_KEYS указываются API-ключи для сервисов Pixabay и Pexels, необходимые для получения изображений.

2. Категории люстр
-  В словаре CATEGORIES определены три стиля:
      - classic (классические люстры)
      - modern (современные люстры)
      - retro (ретро-стиль)
- Для каждого стиля задаются ключевые слова для поиска.

3. Функция download_image(url, folder, filename)
- Скачивает изображение по переданному URL.
- Создаёт нужную папку, если её нет.
- Сохраняет изображение с заданным именем.

4. Функция fetch_pixabay_images()
- Запрашивает изображения с Pixabay по ключевым словам.
- Скачивает изображения, пока не достигнет заданного количества.

5. Функция fetch_pexels_images()

- Запрашивает изображения с Pexels аналогично Pixabay.
- Работает с заголовками API (необходим API-ключ в заголовке запроса).

6. Запуск загрузки изображений
- Для каждой категории (классика, модерн, ретро) создаются соответствующие папки (chandeliers/classic/, chandeliers/modern/, chandeliers/retro/).
- Вызываются функции скачивания для Pixabay и Pexels.
- Количество изображений делится между сервисами.

7. Вывод результата
- После завершения загрузки отображается сообщение «Все изображения загружены и отсортированы по папкам!».

Для скачивание на устройство (ПК, ноутбук) архивируем полученную папку с датасетом.

In [ ]:
!zip -r chandeliers.zip chandeliers/

  adding: chandeliers/ (stored 0%)
  adding: chandeliers/retro/ (stored 0%)
  adding: chandeliers/retro/retro_pexels_243.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pixabay_62.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pixabay_21.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pexels_120.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pexels_191.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pixabay_60.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pixabay_256.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pexels_124.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pixabay_72.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pexels_150.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pixabay_208.jpg (deflated 0%)
  adding: chandeliers/retro/retro_pexels_266.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pixabay_45.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pexels_187.jpg (deflated 1%)
  adding: chandeliers/retro/retro_pixaba

# Удаление лишних данных

К сожалению, при скачивании изображений не гарантируется, что не будет случая, когда могут появиться лишние фотографии, которые будут мешать обучению модели.

Уже вручную удалялись данные (изображения), которые:
* Было нечётко видно на изображении;
* Отсуствовало хорошее освещение (когда фото практически было в темноте);
* Собственно, были не лампы/светильники;
* Дубликаты фотографий.

# Сортировка фотографий

Есть фотографии, которые случайно были засчитаны в другой стиль, поэтому они либо откладывались в сторону, после предыдущего этапа размещались, либо удалялись, если в папке стиля уже было такое изображение.

# Подсчёт общего количества

Количество данных после предыдущих этапов менялся, поэтому посмотрим наглядно, насколько изменилось всё.

Разархивировуем ZIP-файл:

In [ ]:
import zipfile

def extract_zip(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✅ Архив {zip_path} распакован в {extract_to}")

# Пример использования
extract_zip("chandeliers_count_1.zip", "content")

✅ Архив chandeliers_count_1.zip распакован в content


Выполним подсчёт:

In [ ]:
def count_images_in_folders(base_folder):
    for root, dirs, files in os.walk(base_folder):
        image_count = len([f for f in files if f.lower().endswith(('.jpg'))])
        if image_count > 0:
            print(f"📂 {root}: {image_count} фото")

# Пример использования
count_images_in_folders("/content/count_1/chandeliers_count")

📂 /content/count_1/chandeliers_count/retro: 44 фото
📂 /content/count_1/chandeliers_count/modern: 59 фото
📂 /content/count_1/chandeliers_count/classic: 176 фото


Стили имеют общие элементы, но было устроено жёсткое ограничение, общие элементы делились на конкретные категории. По этой причине в ретро и модерне меньше изображений.

# Добавление данных

Постараемся добавить ещё изображений, но только для стилей модерн и ретро. Эти стили очень схожи,и в интерьере люстры/лампы могут чередоваться. Поэтому после скачивания дополнительного датасета, мы в ручную сортируем фотографии между стилями.

In [ ]:
# 🔑 API-ключ Unsplash
UNSPLASH_API_KEY = "(Ваш_API_ключ)"

# 🔍 Функция поиска и скачивания изображений с Unsplash
def fetch_unsplash_images(query, folder_path, num_images):
    headers = {"Authorization": f"Client-ID {UNSPLASH_API_KEY}"}
    image_count = 0
    page = 1

    os.makedirs(folder_path, exist_ok=True)  # Создаём папку, если её нет

    while image_count < num_images:
        url = f"https://api.unsplash.com/search/photos?query={query}&per_page=30&page={page}"
        response = requests.get(url, headers=headers)

        try:
            data = response.json()
            if "results" in data and data["results"]:
                for photo in tqdm(data["results"], desc=f"🔽 Unsplash - {query} (страница {page})"):
                    if image_count >= num_images:
                        break
                    image_url = photo["urls"]["full"]
                    filename = f"{query.replace(' ', '_')}_{image_count+1}.jpg"
                    download_image(image_url, folder_path, filename)
                    image_count += 1
            else:
                print(f"🚫 Нет изображений для {query} на странице {page}")
                break
        except Exception as e:
            print(f"❌ Ошибка Unsplash: {e}")
            break

        page += 1  # Переход на следующую страницу

# 📂 Функция скачивания изображений
def download_image(url, folder, filename):
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        path = os.path.join(folder, filename)
        with open(path, "wb") as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"✅ {filename} загружен!")
    else:
        print(f"❌ Ошибка загрузки {filename}")

# 🔄 Запуск загрузки изображений
fetch_unsplash_images("modern chandeliers", "chandeliers_only_modern", 100)

print("🎉 Все изображения загружены!")

🔽 Unsplash - modern chandeliers (страница 1):   3%|▎         | 1/30 [00:00<00:05,  5.08it/s]

✅ modern_chandeliers_1.jpg загружен!
✅ modern_chandeliers_2.jpg загружен!


🔽 Unsplash - modern chandeliers (страница 1):  13%|█▎        | 4/30 [00:00<00:03,  7.89it/s]

✅ modern_chandeliers_3.jpg загружен!
✅ modern_chandeliers_4.jpg загружен!


🔽 Unsplash - modern chandeliers (страница 1):  20%|██        | 6/30 [00:01<00:07,  3.02it/s]

...

✅ modern_chandeliers_95.jpg загружен!


🔽 Unsplash - modern chandeliers (страница 4):  27%|██▋       | 8/30 [00:00<00:02,  8.22it/s]

✅ modern_chandeliers_96.jpg загружен!
✅ modern_chandeliers_97.jpg загружен!
✅ modern_chandeliers_98.jpg загружен!


🔽 Unsplash - modern chandeliers (страница 4):  33%|███▎      | 10/30 [00:01<00:02,  8.22it/s]

✅ modern_chandeliers_99.jpg загружен!
✅ modern_chandeliers_100.jpg загружен!
🎉 Все изображения загружены!


Повторяем шаги: делаем архив и скачиваем его, сортируем и удаляем лишние данные. После загружаем финальный вариант датасета.

In [ ]:
!zip -r chandeliers_only_modern.zip chandeliers_only_modern/

In [ ]:
import zipfile

def extract_zip(zip_path, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print(f"✅ Архив {zip_path} распакован в {extract_to}")

# Пример использования
extract_zip("/content/chandeliers_final.zip", "content")

✅ Архив /content/chandeliers_final.zip распакован в content


# Обработка и переименование

После скачивания изображения были обработаны и переименованы по следующему алгоритму:
- Составлен список загруженных файлов.
- Применено логичное именование файлов.
- Все файлы распределены по стилям в нужные директории.

Такой метод мы делаем в связи с тем, что раннее мы сортировали лишние данные, поэтому название файла могло отличаться от названия папки, в которой он находится.

In [ ]:
# 📂Укажем путь к папке, файлы в которой нужно переименовать
folder_path = "/content/content/chandeliers/classic"

def rename_files_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ Папка {folder_path} не найдена!")
        return

    files = sorted(os.listdir(folder_path))  # Сортируем файлы по имени
    folder_name = os.path.basename(folder_path)  # Получаем название папки

    for index, file in enumerate(files, start=1):
        old_path = os.path.join(folder_path, file)
        if os.path.isfile(old_path):  # Проверяем, файл ли это (не папка)
            ext = os.path.splitext(file)[1]  # Получаем расширение (например, .jpg, .png)
            new_name = f"{folder_name}_{index}{ext}"  # Формируем новое имя
            new_path = os.path.join(folder_path, new_name)

            os.rename(old_path, new_path)
            print(f"✅ {file} -> {new_name}")

# 🔄 Запуск переименования
rename_files_in_folder(folder_path)

print("🎉 Все файлы переименованы!")

✅ classic_pexels_108.jpg -> classic_1.jpg
✅ classic_pexels_11.jpg -> classic_2.jpg
✅ classic_pexels_12.jpg -> classic_3.jpg
✅ classic_pexels_125.jpg -> classic_4.jpg
✅ classic_pexels_126.jpg -> classic_5.jpg
✅ classic_pexels_128.jpg -> classic_6.jpg
✅ classic_pexels_129.jpg -> classic_7.jpg
✅ classic_pexels_134.jpg -> classic_8.jpg
✅ classic_pexels_135.jpg -> classic_9.jpg
✅ classic_pexels_138.jpg -> classic_10.jpg
✅ classic_pexels_139.jpg -> classic_11.jpg
✅ classic_pexels_142.jpg -> classic_12.jpg
✅ classic_pexels_143.jpg -> classic_13.jpg
✅ classic_pexels_144.jpg -> classic_14.jpg
✅ classic_pexels_146.jpg -> classic_15.jpg
✅ classic_pexels_147.jpg -> classic_16.jpg
✅ classic_pexels_150.jpg -> classic_17.jpg
✅ classic_pexels_158.jpg -> classic_18.jpg
✅ classic_pexels_163.jpg -> classic_19.jpg
✅ classic_pexels_166.jpg -> classic_20.jpg
✅ classic_pexels_171.jpg -> classic_21.jpg
✅ classic_pexels_173.jpg -> classic_22.jpg
✅ classic_pexels_189.jpg -> classic_23.jpg
✅ classic_pexels_195.j

In [ ]:
# 📂 Укажем путь к папке, файлы в которой нужно переименовать
folder_path = "/content/content/chandeliers/modern"

def rename_files_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ Папка {folder_path} не найдена!")
        return

    files = sorted(os.listdir(folder_path))  # Сортируем файлы по имени
    folder_name = os.path.basename(folder_path)  # Получаем название папки

    for index, file in enumerate(files, start=1):
        old_path = os.path.join(folder_path, file)
        if os.path.isfile(old_path):  # Проверяем, файл ли это (не папка)
            ext = os.path.splitext(file)[1]  # Получаем расширение (например, .jpg, .png)
            new_name = f"{folder_name}_{index}{ext}"  # Формируем новое имя
            new_path = os.path.join(folder_path, new_name)

            os.rename(old_path, new_path)
            print(f"✅ {file} -> {new_name}")

# 🔄 Запуск переименования
rename_files_in_folder(folder_path)

print("🎉 Все файлы переименованы!")

✅ classic_pexels_148.jpg -> modern_1.jpg
✅ classic_pexels_156.jpg -> modern_2.jpg
✅ classic_pexels_167.jpg -> modern_3.jpg
✅ classic_pexels_216.jpg -> modern_4.jpg
✅ classic_pexels_86.jpg -> modern_5.jpg
✅ classic_pixabay_148.jpg -> modern_6.jpg
✅ classic_pixabay_168.jpg -> modern_7.jpg
✅ classic_pixabay_82.jpg -> modern_8.jpg
✅ modern_chandeliers_1.jpg -> modern_9.jpg
✅ modern_chandeliers_15.jpg -> modern_10.jpg
✅ modern_chandeliers_17.jpg -> modern_11.jpg
✅ modern_chandeliers_2.jpg -> modern_12.jpg
✅ modern_chandeliers_20.jpg -> modern_13.jpg
✅ modern_chandeliers_22.jpg -> modern_14.jpg
✅ modern_chandeliers_23.jpg -> modern_15.jpg
✅ modern_chandeliers_26.jpg -> modern_16.jpg
✅ modern_chandeliers_27.jpg -> modern_17.jpg
✅ modern_chandeliers_28.jpg -> modern_18.jpg
✅ modern_chandeliers_29.jpg -> modern_19.jpg
✅ modern_chandeliers_3.jpg -> modern_20.jpg
✅ modern_chandeliers_41.jpg -> modern_21.jpg
✅ modern_chandeliers_45.jpg -> modern_22.jpg
✅ modern_chandeliers_46.jpg -> modern_23.jpg


In [ ]:
# 📂 Укажем путь к папке, файлы в которой нужно переименовать
folder_path = "/content/content/chandeliers/retro"

def rename_files_in_folder(folder_path):
    if not os.path.exists(folder_path):
        print(f"❌ Папка {folder_path} не найдена!")
        return

    files = sorted(os.listdir(folder_path))  # Сортируем файлы по имени
    folder_name = os.path.basename(folder_path)  # Получаем название папки

    for index, file in enumerate(files, start=1):
        old_path = os.path.join(folder_path, file)
        if os.path.isfile(old_path):  # Проверяем, файл ли это (не папка)
            ext = os.path.splitext(file)[1]  # Получаем расширение (например, .jpg, .png)
            new_name = f"{folder_name}_{index}{ext}"  # Формируем новое имя
            new_path = os.path.join(folder_path, new_name)

            os.rename(old_path, new_path)
            print(f"✅ {file} -> {new_name}")

# 🔄 Запуск переименования
rename_files_in_folder(folder_path)

print("🎉 Все файлы переименованы!")

✅ classic_pixabay_163.jpg -> retro_1.jpg
✅ classic_pixabay_189.jpg -> retro_2.jpg
✅ classic_pixabay_22.jpg -> retro_3.jpg
✅ classic_pixabay_50.jpg -> retro_4.jpg
✅ modern_chandeliers_11.jpg -> retro_5.jpg
✅ modern_chandeliers_13.jpg -> retro_6.jpg
✅ modern_chandeliers_14.jpg -> retro_7.jpg
✅ modern_chandeliers_16.jpg -> retro_8.jpg
✅ modern_chandeliers_19.jpg -> retro_9.jpg
✅ modern_chandeliers_21.jpg -> retro_10.jpg
✅ modern_chandeliers_32.jpg -> retro_11.jpg
✅ modern_chandeliers_42.jpg -> retro_12.jpg
✅ modern_chandeliers_5.jpg -> retro_13.jpg
✅ modern_chandeliers_58.jpg -> retro_14.jpg
✅ modern_chandeliers_59.jpg -> retro_15.jpg
✅ modern_chandeliers_6.jpg -> retro_16.jpg
✅ modern_chandeliers_61.jpg -> retro_17.jpg
✅ modern_chandeliers_62.jpg -> retro_18.jpg
✅ modern_chandeliers_7.jpg -> retro_19.jpg
✅ modern_chandeliers_8.jpg -> retro_20.jpg
✅ modern_chandeliers_9.jpg -> retro_21.jpg
✅ retro_pexels_105.jpg -> retro_22.jpg
✅ retro_pexels_106.jpg -> retro_23.jpg
✅ retro_pexels_108.jpg

In [ ]:
def count_images_in_folders(base_folder):
    for root, dirs, files in os.walk(base_folder):
        image_count = len([f for f in files if f.lower().endswith(('.jpg'))])
        if image_count > 0:
            print(f"📂 {root}: {image_count} фото")

# Пример использования
count_images_in_folders("/content/content/chandeliers/")

📂 /content/content/chandeliers/retro: 61 фото
📂 /content/content/chandeliers/modern: 79 фото
📂 /content/content/chandeliers/classic: 180 фото


Теперь фотографий в стилях модерн и ретро стало больше. Их количество может подойти для обучения.

# Архивирование

По завершении работы была выполнена архивация всех финальных данных для датасета в ZIP-файл для удобства хранения и дальнейшего использования.

In [ ]:
!zip -r chandeliers_dataset.zip /content/content/chandeliers

  adding: content/content/chandeliers/ (stored 0%)
  adding: content/content/chandeliers/retro/ (stored 0%)
  adding: content/content/chandeliers/retro/retro_19.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_53.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_4.jpg (deflated 0%)
  adding: content/content/chandeliers/retro/retro_21.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_25.jpg (deflated 2%)
  adding: content/content/chandeliers/retro/retro_9.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_23.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_55.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_30.jpg (deflated 2%)
  adding: content/content/chandeliers/retro/retro_16.jpg (deflated 1%)
  adding: content/content/chandeliers/retro/retro_12.jpg (deflated 13%)
  adding: content/content/chandeliers/retro/retro_54.jpg (deflated 1%)
  adding: content/content/chandeliers/ret